# Preflight de datos reales y smoke de loaders multidominio

Esta notebook prepara una muestra pequeña y controlada de A2D2 y UrbanSyn para validar los loaders reales, inferencia semántica, COCOeval y checkpoint/resume. **No ejecuta entrenamiento, no llama backward/optimizer.step y no descarga test.**

Se recupera el artifact de índices ya existente, se reconstruyen los manifiestos remotos completos y después se materializa solo 1 muestra con instancias por split (train/val) y dataset. El límite total de descarga es 2 GiB. No se descarga el conjunto completo.

En Colab Secrets configura `GITHUB_TOKEN` con `Contents: Read-only` y `Actions: Read-only`, limitado al repositorio privado. Nunca pegues el token en el notebook ni en el chat.

## 1. Montar Drive

Los resultados quedarán en `MyDrive/ProcesamientoImagenes/balanceo_cap200`.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, zipfile, hashlib, subprocess, shutil, datetime, requests

drive.mount('/content/drive')
OUT_DRIVE = Path('/content/drive/MyDrive/ProcesamientoImagenes/preflight_datos_reales')
OUT_DRIVE.mkdir(parents=True, exist_ok=True)
print('Destino de informes:', OUT_DRIVE)


## 2. Actualizar repositorio privado

Se requiere el PAT de solo lectura en Contents y Actions. La rama debe ser `urban-models-and-benchmark`.


In [ ]:
from getpass import getpass
REPO='Leohf04/Procesamiento-de-Im-genes-'
BRANCH='urban-models-and-benchmark'
WORKDIR=Path('/content/Procesamiento-de-Im-genes-')
try:
    from google.colab import userdata
    TOKEN=userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN=None
if not TOKEN:
    TOKEN=getpass('GitHub PAT (Contents: Read-only + Actions: Read-only): ')
if not TOKEN:
    raise RuntimeError('Falta el token GitHub.')
askpass=Path('/tmp/colab_git_askpass.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n *Username*) echo x-access-token ;;\n *Password*) echo "$COLAB_GITHUB_TOKEN" ;;\n *) exit 1 ;;\nesac\n')
askpass.chmod(0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','COLAB_GITHUB_TOKEN':TOKEN})
if WORKDIR.exists():
    subprocess.run(['git','-C',str(WORKDIR),'fetch','origin',BRANCH],check=True,env=env)
    subprocess.run(['git','-C',str(WORKDIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH],check=True,env=env)
else:
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,f'https://github.com/{REPO}.git',str(WORKDIR)],check=True,env=env)
del TOKEN,env
os.chdir(WORKDIR)
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('HEAD:',HEAD)


## 3. Recuperar y verificar el artifact existente

El ZIP contiene solo referencias remotas y estadísticas; no contiene RGB ni máscaras. Se verifica el SHA-256 publicado por GitHub antes de extraerlo.


In [ ]:
ARTIFACT_ID=14770171764
RUN_ID=45038801547
EXPECTED_SHA256='b6cbcb54dfd0598e6a9011667073028be1804fc1c112d82e516a42bdbf2ba4e5'
API=f'https://api.github.com/repos/{REPO}/actions/artifacts/{ARTIFACT_ID}'
try:
    from google.colab import userdata
    api_token=userdata.get('GITHUB_TOKEN')
except Exception:
    api_token=None
if not api_token:
    api_token=getpass('GitHub PAT (Fine-grained, Actions: Read-only, acceso a este repo): ')
api_token=api_token.strip()
if not api_token:
    raise RuntimeError('No se recibió ningún token.')
headers={'Authorization':f'Bearer {api_token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
# Diagnosticar por separado autenticación, acceso al repositorio y permiso de Actions.
who=requests.get('https://api.github.com/user',headers=headers,timeout=30)
if who.status_code != 200:
    raise RuntimeError(f'GitHub rechazó la autenticación ({who.status_code}). Comprueba que el PAT no esté revocado/expirado y que sea un token válido. Respuesta: {who.text[:500]}')
repo_check=requests.get(f'https://api.github.com/repos/{REPO}',headers=headers,timeout=30)
if repo_check.status_code != 200:
    raise RuntimeError(f'El token autentica, pero no puede acceder al repositorio privado {REPO} ({repo_check.status_code}). En la configuración del PAT selecciona exclusivamente este repositorio y confirma acceso. Respuesta: {repo_check.text[:500]}')
meta=requests.get(API,headers=headers,timeout=60)
if meta.status_code != 200:
    body={}
    try: body=meta.json()
    except Exception: pass
    accepted=meta.headers.get('X-Accepted-GitHub-Permissions','no indicado')
    if meta.status_code == 403:
        raise RuntimeError('GitHub autentica el PAT y permite ver el repositorio, pero deniega leer Actions/artifacts (403). Solución: edita o crea un Fine-grained PAT: Resource owner = propietario del repositorio; Repository access = Only select repositories y selecciona Procesamiento-de-Im-genes-; Repository permissions > Actions = Read-only. Contents = Read-only puede mantenerse para clonar el repo. Guarda el PAT actualizado en Colab Secrets con el nombre GITHUB_TOKEN, reemplazando el anterior, y vuelve a ejecutar esta celda. Encabezado X-Accepted-GitHub-Permissions: '+accepted+'. Respuesta GitHub: '+str(body)[:700])
    if meta.status_code == 404:
        raise RuntimeError('GitHub no encuentra el artifact o el token no tiene permiso para verlo. Confirma que el PAT tiene Actions: Read-only y que el artifact 14770171764 sigue vigente. Respuesta: '+str(body)[:700])
    raise RuntimeError(f'No se pudo consultar el artifact ({meta.status_code}). X-Accepted-GitHub-Permissions={accepted}. Respuesta: {str(body)[:700]}')
meta_json=meta.json()
if meta_json.get('expired'):
    raise RuntimeError('El artifact de indexación ha expirado; no es posible descargarlo desde GitHub.')
if meta_json.get('workflow_run',{}).get('id') != RUN_ID:
    raise RuntimeError('El artifact no pertenece al run esperado 45038801547.')
r=requests.get(API+'/zip',headers=headers,timeout=300)
if r.status_code != 200:
    raise RuntimeError(f'GitHub encontró el artifact, pero falló la descarga ZIP ({r.status_code}): {r.text[:700]}')
zip_path=Path('/content/urban_annotation_artifact_14770171764.zip')
zip_path.write_bytes(r.content)
actual=hashlib.sha256(zip_path.read_bytes()).hexdigest()
print('Artifact:',meta_json.get('name'),'bytes:',zip_path.stat().st_size)
print('SHA-256:',actual)
if actual != EXPECTED_SHA256:
    raise RuntimeError('El SHA-256 no coincide con el digest verificado del artifact. No se extraerá.')
extract=Path('/content/urban_artifact_14770171764')
if extract.exists(): shutil.rmtree(extract)
extract.mkdir()
with zipfile.ZipFile(zip_path) as zf:
    for info in zf.infolist():
        target=(extract/info.filename).resolve()
        if extract.resolve() not in target.parents and target != extract.resolve():
            raise RuntimeError('Ruta insegura detectada en el ZIP.')
    zf.extractall(extract)
print('Artifact verificado y extraído:',extract)
del api_token,headers


## 4. Reconstruir los seis manifiestos y verificar los índices

El artifact original se generó sin cap (`per_class_cap=null`), por lo que los JSONL de `balanced/` conservan las 1.000/751/783 filas completas. Se usan los archivos semánticos como manifiesto base y se comprueba que las variantes de instancias tienen los mismos IDs y metadatos. Las estadísticas se contrastan fila por fila con los seis índices per-image.


In [ ]:
ART=extract
MANIFEST_DIR=Path('/content/cap200_input/manifests')
INDEX_DIR=ART/'annotation_index'
MANIFEST_DIR.mkdir(parents=True,exist_ok=True)
validation=[]
def read_jsonl(path):
    with path.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
def write_jsonl(path,rows):
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w',encoding='utf-8') as f:
        for row in rows: f.write(json.dumps(row,sort_keys=True,ensure_ascii=False)+'\n')
old_plan=json.loads((ART/'balanced/balanced_training_plan.json').read_text())
if old_plan.get('per_class_cap') is not None:
    raise RuntimeError('El plan de origen no está sin cap; no se puede reconstruir el universo completo.')
for ds in ('a2d2','urbansyn'):
  for split in ('train','val','test'):
    sem=read_jsonl(ART/f'balanced/{ds}_semantic_{split}.jsonl')
    ins=read_jsonl(ART/f'balanced/{ds}_instances_{split}.jsonl')
    ann=read_jsonl(INDEX_DIR/f'{ds}_{split}.annotations.jsonl')
    sem_map={str(x['sample_id']):x for x in sem}
    ins_map={str(x['sample_id']):x for x in ins}
    ann_map={str(x['sample_id']):x for x in ann}
    if len(sem_map)!=len(sem) or len(ins_map)!=len(ins) or len(ann_map)!=len(ann):
        raise RuntimeError(f'{ds}/{split}: IDs duplicados')
    if set(sem_map)!=set(ins_map) or set(sem_map)!=set(ann_map):
        raise RuntimeError(f'{ds}/{split}: IDs de manifiestos/índice no coinciden')
    for sid,row in sem_map.items():
        if any(row.get(k)!=ann_map[sid].get(k) for k in ('semantic_pixels','instance_counts')):
            raise RuntimeError(f'{ds}/{split}: estadísticas no coinciden para {sid}')
        if any(ins_map[sid].get(k)!=row.get(k) for k in ('semantic_pixels','instance_counts')):
            raise RuntimeError(f'{ds}/{split}: estadísticas semántica/instancias distintas para {sid}')
        base_sem={k:v for k,v in row.items() if k not in ('semantic_pixels','instance_counts')}
        base_ins={k:v for k,v in ins_map[sid].items() if k not in ('semantic_pixels','instance_counts')}
        if base_sem!=base_ins:
            raise RuntimeError(f'{ds}/{split}: metadatos de manifiesto distintos para {sid}')
    base=[{k:v for k,v in row.items() if k not in ('semantic_pixels','instance_counts')} for row in sem]
    # Restore the exact manifest order used by prepare_reproducible_subsets.py.
    base.sort(key=lambda row: hashlib.sha256(f"2026:{row['sample_id']}".encode('utf-8')).hexdigest())
    write_jsonl(MANIFEST_DIR/f'{ds}_{split}.jsonl',base)
    validation.append({'dataset':ds,'split':split,'manifest_rows':len(base),'index_rows':len(ann),'unique_ids':len(sem_map),'ids_match':True,'stats_match':True,'task_manifests_match':True})
if len(validation)!=6 or any(x['manifest_rows']!=x['index_rows'] for x in validation):
    raise RuntimeError('Validación de reconstrucción incompleta.')
validation_path=Path('/content/cap200_input/artifact_reconstruction_validation.json')
validation_path.parent.mkdir(parents=True,exist_ok=True)
validation_path.write_text(json.dumps({'artifact_id':ARTIFACT_ID,'source_run':RUN_ID,'checks':validation},indent=2,ensure_ascii=False)+'\n')
print(json.dumps(validation,indent=2,ensure_ascii=False))


## 5. Preflight remoto limitado a una muestra por split

Consulta tamaños de los archivos remotos sin descargar RGB ni máscaras. Se selecciona la primera muestra con instancias en cada combinación dataset/split para que también se pueda probar Mask R-CNN y COCOeval. Límite de seguridad: 2 GiB.

In [ ]:
from importlib.util import find_spec
missing = [p for p in ('boto3','huggingface_hub') if find_spec(p) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
DATA_DIR = Path('/content/urban_data_preflight')
if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)
preflight_cmd = [sys.executable,'scripts/materialize_urban_experiment_data.py',
 '--manifest-dir',str(MANIFEST_DIR),'--annotation-index-dir',str(INDEX_DIR),
 '--output-dir',str(DATA_DIR),'--dry-run','--max-samples-per-split','1',
 '--require-instances','--max-total-bytes',str(2*1024**3)]
pre = subprocess.run(preflight_cmd,cwd=WORKDIR,text=True,capture_output=True)
print(pre.stdout)
if pre.returncode:
    print(pre.stderr)
    raise RuntimeError('Falló el preflight remoto; no se descargaron datos.')
preflight_report = json.loads((DATA_DIR/'materialization_preflight.json').read_text())
if preflight_report.get('status')!='PREFLIGHT_PASS' or preflight_report.get('test_touched') is not False:
    raise RuntimeError('El informe no confirma preflight seguro.')
shutil.copy2(DATA_DIR/'materialization_preflight.json',OUT_DRIVE/'materialization_preflight.json')
print('Estimación de muestra:',preflight_report['estimated_remote_asset_gib'],'GiB')
print('Activos:',preflight_report['asset_count'])
print('Informe guardado en Drive:',OUT_DRIVE/'materialization_preflight.json')


## 6. Materializar solo cuatro muestras (train/val)

Descarga una muestra train y una val por dataset, ambas con al menos una instancia anotada. No toca test. Los archivos quedan en el almacenamiento temporal de Colab, no en el teléfono ni en Git.

In [ ]:
materialize_cmd = [sys.executable,'scripts/materialize_urban_experiment_data.py',
 '--manifest-dir',str(MANIFEST_DIR),'--annotation-index-dir',str(INDEX_DIR),
 '--output-dir',str(DATA_DIR),'--max-samples-per-split','1','--require-instances',
 '--max-total-bytes',str(2*1024**3)]
mat = subprocess.run(materialize_cmd,cwd=WORKDIR,text=True,capture_output=True)
print(mat.stdout)
if mat.returncode:
    print('ERROR:',mat.stderr)
    raise RuntimeError('Falló la materialización limitada.')
materialization_report = json.loads((DATA_DIR/'materialization_report.json').read_text())
if materialization_report.get('status')!='PASS' or materialization_report.get('test_touched') is not False:
    raise RuntimeError('Materialización no confirmó PASS o tocó test.')
for ds in ('a2d2','urbansyn'):
  for split in ('train','val'):
    local = read_jsonl(DATA_DIR/'local_manifests'/f'{ds}_{split}.jsonl')
    ann = read_jsonl(DATA_DIR/'local_annotation_index'/f'{ds}_{split}.annotations.jsonl')
    if len(local)!=1 or len(ann)!=1 or local[0]['sample_id']!=ann[0]['sample_id']:
        raise RuntimeError(f'{ds}/{split}: manifiesto local e índice no coinciden')
shutil.copy2(DATA_DIR/'materialization_report.json',OUT_DRIVE/'materialization_report.json')
print('Muestra materializada. Directorio local:',DATA_DIR)
print('Índices locales:',DATA_DIR/'local_annotation_index')
print('Informe guardado en Drive:',OUT_DRIVE/'materialization_report.json')


## 7. Smoke de los cuatro loaders, inferencia, COCOeval y checkpoint/resume

Ejecuta las cuatro combinaciones de dataset/tarea con una muestra train y una val. Comprueba targets, tensores finitos, forward semántico, métricas COCO Box/Mask AP con la muestra real de validación y round-trip de checkpoint restaurando RNG del loader. **No hay backward ni optimizer.step.** Usa CPU para limitar consumo de VRAM.

In [ ]:
if find_spec('pycocotools') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','pycocotools'],check=True)
smoke_report = OUT_DRIVE/'urban_real_data_preflight_smoke.json'
smoke_stdout = OUT_DRIVE/'urban_real_data_preflight_smoke.stdout.log'
smoke_stderr = OUT_DRIVE/'urban_real_data_preflight_smoke.stderr.log'
for p in (smoke_report,smoke_stdout,smoke_stderr):
    if p.exists(): p.unlink()
smoke_cmd = [sys.executable,'-u','scripts/smoke_test_urban_experiment_factory.py',
 '--data-dir',str(DATA_DIR),'--annotation-index-dir',str(DATA_DIR/'local_annotation_index'),
 '--output',str(smoke_report),'--size','256']
smoke = subprocess.run(smoke_cmd,cwd=WORKDIR,text=True,capture_output=True)
smoke_stdout.write_text(smoke.stdout or '',encoding='utf-8')
smoke_stderr.write_text(smoke.stderr or '',encoding='utf-8')
print(smoke.stdout[-18000:])
if smoke.returncode:
    print('ERROR REAL:',smoke.stderr[-12000:])
    raise RuntimeError('Falló el smoke de factory/COCOeval/checkpoint. Logs guardados en Drive.')
result = json.loads(smoke_report.read_text())
if result.get('status')!='PASS' or result.get('training_started') is not False or result.get('optimizer_steps')!=0 or result.get('backward_calls')!=0:
    raise RuntimeError('El informe no confirma PASS sin entrenamiento.')
print('PASS. Informe:',smoke_report)
print('Logs:',smoke_stdout,smoke_stderr)
print('No se entrenó ningún modelo y test no se descargó.')


## 8. Resultado y siguiente paso

Comparte el JSON `urban_real_data_preflight_smoke.json` o el bloque de salida completo. Si pasa, la siguiente puerta será estimar el tamaño de materialización completa, verificar almacenamiento persistente y preparar train/validation completos. No ejecutes las secciones de entrenamiento controlado del notebook antiguo ni cambies `RUN_TRAINING`.